# Text Mining Project — Final Solution
**Spring Semester 2025/2026 | NOVA IMS**

**Model:** FinBERT Fine-tuned (end-to-end Transformer Encoder)

**Pipeline:** Raw text → `preprocess_transformer` → FinBERT tokenizer → fine-tune on full train → predict `test.csv` → `pred_xx.csv`

---

## 1. Setup & Imports

In [ ]:
!pip install transformers torch datasets -q

In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

from tqdm import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns

MODEL_NAME = 'ProsusAI/finbert'
SEED       = 42
MAX_LEN    = 128

print('✅ Imports OK')

## 2. Load Data

In [ ]:
df_train = pd.read_csv('train.csv')
df_test  = pd.read_csv('test.csv')

print(f'Train shape: {df_train.shape}')
print(f'Test shape:  {df_test.shape}')
print(f'\nClass distribution (train):')
print(df_train['label'].value_counts().sort_index())
df_train.head()

## 3. Preprocessing

Minimal cleaning for Transformer models — FinBERT's tokenizer handles sub-word splitting.
Heavy cleaning (stopwords, stemming) would hurt performance on transformer models.

In [ ]:
def preprocess_transformer(text_list):
    """
    Minimal cleaning for Transformer models (FinBERT).
    Steps: remove URLs → normalise whitespace.
    Symbols, punctuation and casing are preserved — FinBERT tokenizer handles them.
    """
    results = []
    for text in tqdm(text_list, desc='Preprocessing'):
        text = str(text)
        text = re.sub(r'http\S+|www\S+', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        results.append(text)
    return results


print('Preprocessing train set...')
X_train = preprocess_transformer(df_train['text'].tolist())
y_train = df_train['label'].tolist()

print('Preprocessing test set...')
X_test = preprocess_transformer(df_test['text'].tolist())

print(f'\n✅ Done. Train: {len(X_train)} tweets | Test: {len(X_test)} tweets')
print(f'\nExample (original):    {df_train["text"].iloc[0]}')
print(f'Example (preprocessed): {X_train[0]}')

## 4. Tokenization

FinBERT tokenizer converts each tweet to input IDs + attention masks.
`max_length=128` covers >95% of tweets without truncation.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LEN
    )

# Build HuggingFace Datasets
train_ds = Dataset.from_dict({'text': X_train, 'label': y_train})
test_ds  = Dataset.from_dict({'text': X_test})

train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

print(f'Train dataset: {train_ds}')
print(f'Test dataset:  {test_ds}')
print('\n✅ Tokenization complete.')

## 5. Fine-tune FinBERT on Full Training Set

Training on the **full** training set (9543 tweets) — no validation split here.
Model selection was done in `tm_tests_xx` via Stratified K-Fold CV.

**Hyperparameters** (tuned in `tm_tests_xx`):
- 1 epoch — best validation loss was epoch 1 in CV (early stopping criterion)
- `weight_decay=0.01` — regularisation to reduce overfitting
- `warmup_ratio=0.1` — gradual learning rate warm-up

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

training_args = TrainingArguments(
    output_dir='./finbert_final',
    num_train_epochs=1,          # epoch 1 had best val loss in tm_tests_xx
    per_device_train_batch_size=16,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy='no',          # no checkpoints needed — full train run
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
)

print('Starting fine-tuning on full training set...')
print('(Expected time: ~50 min on CPU, ~5 min on GPU)\n')
trainer.train()
print('\n✅ Fine-tuning complete.')

## 6. Generate Predictions on test.csv

In [ ]:
print('Generating predictions on test set...')
preds_output = trainer.predict(test_ds)
final_preds  = preds_output.predictions.argmax(axis=1)

print(f'\nPrediction distribution:')
pred_series = pd.Series(final_preds)
for label, count in pred_series.value_counts().sort_index().items():
    names = {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}
    print(f'  {names[label]} ({label}): {count} ({count/len(final_preds)*100:.1f}%)')

print('\n✅ Predictions generated.')

## 7. Save Submission File

In [ ]:
submission = pd.DataFrame({
    'id':    df_test['id'],
    'label': final_preds
})

submission.to_csv('pred_xx.csv', index=False)

print('✅ Submission saved: pred_xx.csv')
print(f'   Shape: {submission.shape}')
print()
print(submission.head(10).to_string(index=False))